# Demo 03 : One task at three levels

Run the cells in order and watch. There is nothing to write.

Run the next cell first. It loads the course key and builds `llm`, the course model. Nothing below it depends on where the notebook runs.

In [1]:
# Course setup : run this cell first.
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph-Student/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.6", "langchain-core==1.6.12", "langchain-openai==1.7.1", "langgraph==1.2.17", "langgraph-checkpoint-sqlite==3.1.2", "deepagents[quickjs]==0.7.30", "langchain-quickjs==0.3.9", "langsmith==0.14.12", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    # OpenRouter has no Luna endpoint that honours temperature or parallel_tool_calls ;
    # seed is honoured, and routing prefers OpenAI so repeated runs hit one provider
    THINKING_OFF = {"reasoning": {"effort": "none"}, "provider": {"order": ["openai"]}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# One shared limiter paces every model call in this process ; COURSE_RPS sets the budget.
# Paid OpenRouter models have no platform request cap (measured 26 October 2026)
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "2")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    if PROVIDER == "ollama":
        kw.setdefault("temperature", 0)
    else:
        kw.setdefault("seed", 7)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

model openai/gpt-6-luna via course


In [2]:
import json
import re
import sqlite3
from pathlib import Path

from langchain.tools import tool

DB = data("game-studio/studio.db")
SECTIONS = []
for path in sorted(Path(data_folder("game-studio/docs")).glob("*.md")):
    title, *parts = path.read_text(encoding="utf-8").split("\n## ")
    SECTIONS += [f"{title.splitlines()[0]}\n## {part}" for part in parts]


def rows(sql, *args):
    with sqlite3.connect(DB) as conn:
        conn.row_factory = sqlite3.Row
        return [dict(row) for row in conn.execute(sql, args)]


def words(text):
    return set(re.findall(r"[a-z0-9-]{3,}", text.lower()))


def best(texts, query, k):
    q = words(query)
    return sorted(texts, key=lambda t: -len(q & words(t)))[:k]

We define six tools over the studio data, a meter for calls and tokens, and the task with the facts a correct
answer contains.

In [3]:
@tool
def game_info(title: str) -> str:
    """Look up a Comet Arcade game by its title or id. Returns id, title, genre, status and pitch."""
    t = title.strip()
    found = rows("SELECT game_id, title, genre, status, pitch FROM games WHERE game_id = ? OR title LIKE ?", t.upper(), f"%{t}%")
    return json.dumps(found[0]) if found else "Unknown game"

@tool
def bugs_for_game(game_id: str, status: str = "open") -> str:
    """List bugs for a game id such as CS-01. Status is open, fixed or all."""
    status = status.strip().lower()
    return json.dumps(rows("SELECT bug_id, title, severity, status, found_by FROM bugs WHERE game_id = ? AND (? = 'all' OR status = ?)",
                           game_id.strip().upper(), status, status))

@tool
def bug_details(bug_id: str) -> str:
    """Read one bug by id such as BUG-101, with severity, status and notes."""
    found = rows("SELECT * FROM bugs WHERE bug_id = ?", bug_id.strip().upper())
    return json.dumps(found[0]) if found else "Unknown bug"

@tool
def builds_for_game(game_id: str) -> str:
    """List the builds of a game id such as CS-01, with tests passed and status."""
    return json.dumps(rows("SELECT build, twist, tests_passed, tests_total, status, notes FROM builds WHERE game_id = ? ORDER BY build",
                           game_id.strip().upper()))

@tool
def assets_for_game(game_id: str) -> str:
    """List sprites, palettes and sounds for a game id such as CS-01, with colour and status."""
    return json.dumps(rows("SELECT asset_id, kind, name, colour, status FROM assets WHERE game_id = ?", game_id.strip().upper()))

@tool
def search_design_docs(query: str) -> str:
    """Search the design docs and the studio style guide. Returns the three best matching sections."""
    return "\n\n---\n\n".join(best(SECTIONS, query, 3))

from langchain_core.callbacks import BaseCallbackHandler


class Meter(BaseCallbackHandler):
    def __init__(self):
        self.calls = self.input = self.output = 0

    def on_llm_end(self, response, **kwargs):
        usage = response.generations[0][0].message.usage_metadata or {}
        self.calls += 1
        self.input += usage.get("input_tokens", 0)
        self.output += usage.get("output_tokens", 0)

    def __str__(self):
        return f"{self.calls} calls, {self.input:,} input tokens, {self.output:,} output tokens"


def correct(answer, expected):
    text = answer.lower()
    return all(any(alt in text for alt in item.split("|")) for item in expected)

TASK = "BUG-104 is open on Comet Snake. How serious is it, and how many tests does the latest build pass?"
EXPECTED = ["major", "19"]
PROMPT = "You are the studio assistant of Comet Arcade. Answer from the tools only. Be brief."
TOOL_LIST = [game_info, bugs_for_game, bug_details, builds_for_game, assets_for_game, search_design_docs]
results = {}

## a · graph-by-hand

The graph fixes the steps. The model extracts the bug id, plain Python looks up the bug, the game and the latest
build, and the model writes the answer.

In [4]:
def look_up_bug(bug_id: str) -> dict:
    bug = rows("SELECT * FROM bugs WHERE bug_id = ?", bug_id)[0]
    game = rows("SELECT title, status FROM games WHERE game_id = ?", bug["game_id"])[0]
    builds = rows("SELECT build, tests_passed, tests_total, status, notes FROM builds WHERE game_id = ? ORDER BY build",
                  bug["game_id"])
    return {"bug": bug, "game": game, "latest_build": builds[-1]}

from typing import TypedDict

from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel


class BugId(BaseModel):
    bug_id: str


class Desk(TypedDict, total=False):
    question: str
    bug_id: str
    facts: dict
    answer: str


def extract(state: Desk) -> dict:
    found = llm.with_structured_output(BugId).invoke(f"Extract the bug id.\n\n{state['question']}")
    return {"bug_id": found.bug_id}


def look_up(state: Desk) -> dict:
    return {"facts": look_up_bug(state["bug_id"])}


def answer(state: Desk) -> dict:
    reply = llm.invoke(f"Answer the question from the facts. Be brief.\n\n"
                       f"Question : {state['question']}\nFacts : {state['facts']}")
    return {"answer": reply.content}


builder = StateGraph(Desk)
builder.add_node("extract", extract)
builder.add_node("look_up", look_up)
builder.add_node("answer", answer)
builder.add_edge(START, "extract")
builder.add_edge("extract", "look_up")
builder.add_edge("look_up", "answer")
builder.add_edge("answer", END)
graph = builder.compile()

meter = Meter()
result = graph.invoke({"question": TASK}, {"callbacks": [meter]})
print("facts :", result["facts"])
print("answer:", result["answer"])
print(meter)
results["graph by hand"] = (meter, correct(result["answer"], EXPECTED))

facts : {'bug': {'bug_id': 'BUG-104', 'game_id': 'CS-01', 'title': 'Shield pickup does nothing', 'severity': 'major', 'status': 'open', 'found_by': 'Tester', 'notes': 'Shield extras stay on the board and the shield flag stays false.'}, 'game': {'title': 'Comet Snake', 'status': 'in development'}, 'latest_build': {'build': 3, 'tests_passed': 19, 'tests_total': 19, 'status': 'testing', 'notes': 'All rule tests pass, comet contrast open'}}
answer: BUG-104 is a major issue. The latest build passes 19 of 19 tests.
2 calls, 243 input tokens, 39 output tokens


## b · create-agent

`create_agent` builds the tool-calling loop for us. The model decides which tools to call and in what order.

In [5]:
from langchain.agents import create_agent

agent = create_agent(llm, tools=TOOL_LIST, system_prompt=PROMPT)
meter = Meter()
messages = agent.invoke({"messages": [{"role": "user", "content": TASK}]}, {"callbacks": [meter]})["messages"]
print("tools :", [m.name for m in messages if m.type == "tool"])
print("answer:", messages[-1].content)
print(meter)
results["create_agent"] = (meter, correct(messages[-1].content, EXPECTED))

tools : ['bug_details', 'game_info', 'builds_for_game']
answer: BUG-104 is **major**. The latest Comet Snake build (Build 3) passes **19 tests**.
3 calls, 1,433 input tokens, 106 output tokens


We inspect the compiled graph. Its model and tool nodes form the agent loop.

In [6]:
print(list(agent.get_graph().nodes))

['__start__', 'model', 'tools', '__end__']


## c · deep-agent

We pass the same tools and prompt to `create_deep_agent`. Version 0.7.30 adds filesystem tools, a general-purpose
subagent and summarisation. Planning with a todo list requires additional middleware.

In [7]:
from deepagents import create_deep_agent

deep = create_deep_agent(model=llm, tools=TOOL_LIST, system_prompt=PROMPT)
meter = Meter()
messages = deep.invoke({"messages": [{"role": "user", "content": TASK}]}, {"callbacks": [meter]})["messages"]
print("tools :", [m.name for m in messages if m.type == "tool"])
print("answer:", messages[-1].content)
print(meter)
results["create_deep_agent"] = (meter, correct(messages[-1].content, EXPECTED))
print(list(deep.get_graph().nodes))

tools : ['bug_details', 'builds_for_game']
answer: BUG-104 is **major**. The latest build (build 3) passes **19 tests**.
2 calls, 4,797 input tokens, 83 output tokens
['__start__', 'model', 'tools', '_ParentNotifierMiddleware.after_agent', 'PatchToolCallsMiddleware.before_agent', '__end__']


We compare the three runs on correctness, calls and tokens. The deep agent receives additional tool descriptions and
middleware instructions. This short lookup measures their overhead. It does not measure their value on a longer task.

In [8]:
print(f"{'level':18} {'correct':>7} {'calls':>6} {'input':>8} {'output':>7}")
for name, (m, ok) in results.items():
    print(f"{name:18} {str(ok):>7} {m.calls:>6} {m.input:>8,} {m.output:>7}")

level              correct  calls    input  output
graph by hand         True      2      243      39
create_agent          True      3    1,433     106
create_deep_agent     True      2    4,797      83


## Things to try

Optional.

- Ask the three a question the graph was not built for, such as which assets of Comet Snake are still drafts
- Give the deep agent a task over all six games and inspect its tool calls